# Discrete DCP — scikit-learn, LightGBM, and quantile-forest

DCP for count data with randomized PIT, an integer quantile grid, CDF, PMF, and optimal inventory.

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split
from lightgbm import LGBMRegressor
from quantile_forest import RandomForestQuantileRegressor
import sys
import os
sys.path.append(os.path.abspath("../.."))
from tinyconformal.distribution import DiscreteDistributionalConformalPredictiveSystem
from tinyconformal.utils import MultiQuantileRegressor, NewsvendorSolver

rng = np.random.default_rng(42)
X = rng.uniform(0, 3, size=(3500, 1))
mu = np.exp(0.5 + 0.65 * X[:, 0])
y = rng.poisson(mu)
X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.4, random_state=42)
X_cal, X_test, y_cal, y_test = train_test_split(X_tmp, y_tmp, test_size=0.5, random_state=42)
levels = (0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99)

In [2]:
models = {
    "scikit-learn": MultiQuantileRegressor(
        HistGradientBoostingRegressor(loss="quantile", max_iter=200, random_state=42), quantiles=levels
    ),
    "LightGBM": MultiQuantileRegressor(
        LGBMRegressor(objective="quantile", n_estimators=200, verbosity=-1, random_state=42), quantiles=levels
    ),
    "quantile-forest": RandomForestQuantileRegressor(
        n_estimators=250, min_samples_leaf=8, random_state=42, n_jobs=-1
    ),
}
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    dcp = DiscreteDistributionalConformalPredictiveSystem(
        model, quantiles=levels, minimum=0, random_state=42
    ).fit(X_cal, y_cal)
    distribution = dcp.predict_distribution(X_test)
    results[name] = (dcp, distribution)
    display(distribution.evaluate(y_test))

,coverage,empirical_coverage,mean_width,winkler_score
0,0.50,0.545714,2.482857,6.157143
1,0.80,0.848571,5.877143,8.191429
2,0.90,0.965714,8.341429,9.455714
3,0.95,0.981429,9.187143,10.387143


,coverage,empirical_coverage,mean_width,winkler_score
0,0.50,0.557143,2.618571,6.144286
1,0.80,0.841429,5.801429,8.244286
2,0.90,0.950000,7.868571,9.468571
3,0.95,0.950000,7.932857,11.132857


,coverage,empirical_coverage,mean_width,winkler_score
0,0.50,0.622857,3.088571,5.825714
1,0.80,0.858571,5.818571,8.161429
2,0.90,0.892857,6.821429,9.964286
3,0.95,0.964286,8.607143,10.492857


## CDF, PMF, PPF, and quantiles

In [3]:
distribution = results["quantile-forest"][1]
requested = np.array([0.1, 0.5, 0.9])
requested_matrix = np.broadcast_to(requested, (len(distribution), len(requested)))
quantile_predictions = distribution.ppf(requested_matrix)

pd.DataFrame({
    "y": y_test[:10],
    "cdf_at_y": distribution.cdf(y_test)[:10],
    "pmf_at_y": distribution.pmf(y_test)[:10],
    "q10": quantile_predictions[:10, 0],
    "q50": quantile_predictions[:10, 1],
    "q90": quantile_predictions[:10, 2],
})

,y,cdf_at_y,pmf_at_y,q10,q50,q90
0,2,0.657632,0.447932,0,2,4
1,9,0.412268,0.000000,7,10,14
2,3,0.957204,0.052782,0,1,2
3,4,0.657632,0.000000,1,3,6
4,2,0.209700,0.161198,2,4,7
5,1,0.412268,0.202568,0,2,4
6,9,1.000000,0.174037,2,5,9
7,8,1.000000,0.000000,1,3,5
8,3,0.209700,0.104137,2,6,9
9,10,0.825963,0.168331,5,8,12


## Inventory solver

In [4]:
decision_frame = pd.DataFrame({
    "unique_id": np.arange(len(y_test)).astype(str),
    "ds": pd.Timestamp("2026-01-01"),
    "shortage_cost": 9.0,
    "holding_cost": 1.0,
})
stock = NewsvendorSolver.optimize_distribution(
    decision_frame,
    distribution,
    underage_cost="shortage_cost",
    overage_cost="holding_cost",
)
assert np.all(stock["y_optimal"] == np.floor(stock["y_optimal"]))
stock.head()

,unique_id,ds,shortage_cost,holding_cost,critical_ratio,y_optimal
0,0,2026-01-01,9.0,1.0,0.9,4.0
1,1,2026-01-01,9.0,1.0,0.9,14.0
2,2,2026-01-01,9.0,1.0,0.9,2.0
3,3,2026-01-01,9.0,1.0,0.9,6.0
4,4,2026-01-01,9.0,1.0,0.9,7.0


### Selecting inventory units with `max_k` or `units`

Use `max_k` for the dense grid from zero through the requested limit. Use `units` for a sparse or stepped grid; the two arguments are mutually exclusive.

In [5]:
pmf_dense = NewsvendorSolver.pmf_distribution(
    decision_frame, distribution, max_k=10
)
pmf_sparse = NewsvendorSolver.pmf_distribution(
    decision_frame, distribution, units=range(0, 21, 5)
)
display(pmf_dense[["unique_id", "P(Y=0)", "P(Y=5)", "P(Y=10)", "P(Y>10)"]].head())
display(pmf_sparse[["unique_id", "P(Y=0)", "P(Y=5)", "P(Y=10)", "P(Y=15)", "P(Y=20)"]].head())

,unique_id,P(Y=0),P(Y=5),P(Y=10),P(Y>10)
0,0,0.085592,0.000000,0.000000,0.019971
1,1,0.000000,0.000000,0.000000,0.607703
2,2,0.392297,0.000000,0.000000,0.019971
3,3,0.000000,0.000000,0.042796,0.019971
4,4,0.000000,0.168331,0.000000,0.019971


,unique_id,P(Y=0),P(Y=5),P(Y=10),P(Y=15),P(Y=20)
0,0,0.085592,0.000000,0.000000,0.000000,0.0
1,1,0.000000,0.000000,0.000000,0.052782,0.0
2,2,0.392297,0.000000,0.000000,0.000000,0.0
3,3,0.000000,0.000000,0.042796,0.000000,0.0
4,4,0.000000,0.168331,0.000000,0.000000,0.0


In [6]:
marginal_dense = NewsvendorSolver.marginal_benefit_distribution(
    decision_frame,
    distribution,
    underage_cost="shortage_cost",
    overage_cost="holding_cost",
    max_k=10,
)
marginal_sparse = NewsvendorSolver.marginal_benefit_distribution(
    decision_frame,
    distribution,
    underage_cost="shortage_cost",
    overage_cost="holding_cost",
    units=[0, 5, 10, 15, 20],
)
display(marginal_dense[["unique_id", "MB(k=0)", "MB(k=5)", "MB(k=10)"]].head())
display(marginal_sparse[["unique_id", "MB(k=0)", "MB(k=5)", "MB(k=10)", "MB(k=15)", "MB(k=20)"]].head())

,unique_id,MB(k=0),MB(k=5),MB(k=10)
0,0,8.800285,-1.000000,-1.000000
1,1,8.800285,8.800285,4.877318
2,2,8.800285,-1.000000,-1.000000
3,3,8.800285,2.423680,-0.572040
4,4,8.800285,2.423680,-1.000000


,unique_id,MB(k=0),MB(k=5),MB(k=10),MB(k=15),MB(k=20)
0,0,8.800285,-1.000000,-1.000000,-1.000000,-1.0
1,1,8.800285,8.800285,4.877318,-0.044223,-1.0
2,2,8.800285,-1.000000,-1.000000,-1.000000,-1.0
3,3,8.800285,2.423680,-0.572040,-1.000000,-1.0
4,4,8.800285,2.423680,-1.000000,-1.000000,-1.0
